# 07C 比较一次改动，并复盘失败

我们在第六节同一个 Harness 上比较 base 和 condition_check。后者只增加一条条件核查提示；不同时更换模型、工具或资料。

每题每版本计划两次，独立状态，交替先后顺序。默认读取全部已保存记录，真实请求必须显式运行课程单元 CLI。

**本实践在评估链中的位置**：把 07A 的任务契约和 07B 的指标落到一次真实受控改动，保留失败、成本和最早证据偏离点，不用单个总分掩盖原因。

**贯穿案例**

对同一模型、工具、资料与预算，只改回答前的条件核对提示，再按任务和重复次数配对比较保存的试验。`summarize` 汇总通过率，逐题诊断仍保留格式、事实和引用失败；结论只能归于这组受控任务，不能从几次结果推出通用模型能力。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：逐题配对看改动，保留失败在分母里；用真实 Trial 计算配对差异和区间。

课件页：L07.06-S01、L07.06-S02、L07.06-S03、L07.07-S01、L07.07-S06、L07.07-S02。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, copy, math, statistics, hashlib
from collections import defaultdict
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "07 智能体评估与对照实验"
sys.path.insert(0, str(CHAPTER / "代码"))
from evaluation_lab import parse_answer, typed_equal, observed_files, grade_trial, pass_metrics, summarize, diagnose_format
DATASET = json.loads((CHAPTER / "数据/tasks.json").read_text(encoding='utf-8'))
TASKS = {task["id"]:task for task in DATASET["tasks"]}
print("四项开发集任务；真实请求默认关闭。")


四项开发集任务；真实请求默认关闭。


## 1. 检查究竟改了什么

增加一条提示，可能改变模型后续行为、请求次数和回答长度。这些是改动产生的结果，不是需要强行保持一致的量。

固定的是输入任务、资料、模型、允许工具和上限。不要在看到结果后丢掉失败试验，再声称新版本更好。


In [2]:
record = json.loads((CHAPTER / "数据/实验记录/deepseek-evaluation.json").read_text(encoding='utf-8'))
print("唯一预设改动:", record["single_change"])
print("固定上限:", record["limits"], "每次最大输出:", record["max_tokens_per_call"])
snapshot_checks = {
    relative: hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == digest
    for relative, digest in record["source_snapshot"].items()
}
unchanged_checks = {path: ok for path, ok in snapshot_checks.items()
                    if path not in record["adapted_student_files"]}
assert unchanged_checks and all(unchanged_checks.values())
print("未改动的示例文件与历史快照一致；学生版 Skill 已调整，历史哈希单独保留。")


唯一预设改动: 回答前逐项核对运行条件：区分源码保证与样例文件；区分相对路径与绝对路径；没有检查的类型和字段不要声称有保证。
固定上限: {'max_turns': 4, 'max_tools': 8, 'context_chars': 18000} 每次最大输出: 1000


未改动的示例文件与历史快照一致；学生版 Skill 已调整，历史哈希单独保留。


## 2. 保留停止原因、格式失败和实际内容

最先几次真实回答在 JSON 前后添加了解释。题目明确要求只输出 JSON，因此严格格式评分失败，但这不等于里面的每个事实都错了。

原评分器无法解析整段文本时，facts 和 citations 的 False 表示“未获得通过证据”，不能解释成已经逐项确认事实错误。下面读取原文，然后用单独的诊断提取检查唯一 JSON 围栏。它不改变原任务的格式失败判定，也不核验围栏外的文字。

另有请求未正常完成时，保留 needs_review 和已用次数。没有返回的 usage 是未知值，不应记作零成本。


In [3]:
for trial in record["trials"]:
    print(trial["task_id"], trial["repeat"], trial["variant"], trial["state"]["status"], trial["grade"]["passed"])
print("第一份原始回答:")
print(record["trials"][0]["state"]["answer"])


relative-base 1 base final_answer True
relative-base 1 condition_check final_answer True
missing-file 1 condition_check final_answer False
missing-file 1 base final_answer False
json-contract 1 base final_answer True
json-contract 1 condition_check final_answer False
absolute-env 1 condition_check final_answer True
absolute-env 1 base final_answer True
relative-base 2 condition_check final_answer True
relative-base 2 base final_answer True
missing-file 2 base final_answer False
missing-file 2 condition_check tool_budget False
json-contract 2 condition_check final_answer True
json-contract 2 base final_answer True
absolute-env 2 base final_answer True
absolute-env 2 condition_check final_answer True
第一份原始回答:
{
  "answer": {
    "default_path": "settings.json",
    "relative_base": "process_cwd"
  },
  "evidence": [
    {"path": "config.py", "start": 7, "end": 11},
    {"path": "config.py", "start": 8, "end": 8},
    {"path": "main.py", "start": 2, "end": 7},
    {"path": "settings.json"

### 用真实 Trial 计算配对差异和区间

配套课件：L07.06-S03、L07.07-S01、L07.07-S06。

对每道题、每次重复把基础版和条件核对版配成一对。用精确二项方法给出通过率区间，再对不一致的配对做符号检验；样本很小，区间会很宽，不能宣称普遍提升。

In [4]:
from collections import defaultdict
from scipy.stats import binomtest
case_pairs = defaultdict(dict)
for trial in record["trials"]:
    case_pairs[(trial["task_id"], trial["repeat"])][trial["variant"]] = bool(trial["grade"]["passed"])
assert all(set(pair) == {"base", "condition_check"} for pair in case_pairs.values())
case_wins = sum(p["condition_check"] and not p["base"] for p in case_pairs.values())
case_losses = sum(p["base"] and not p["condition_check"] for p in case_pairs.values())
print("配对数:", len(case_pairs), "改进版独赢:", case_wins, "基础版独赢:", case_losses)
for variant in ["base", "condition_check"]:
    passed = sum(p[variant] for p in case_pairs.values())
    interval = binomtest(passed, len(case_pairs)).proportion_ci(method="exact")
    print(variant, "通过:", passed, "/", len(case_pairs), "95%精确区间:",
          (round(interval.low, 3), round(interval.high, 3)))
discordant = case_wins + case_losses
print("不一致对的双侧符号检验 p:",
      round(binomtest(case_wins, discordant, 0.5).pvalue, 4) if discordant else "无不一致对")


配对数: 8 改进版独赢: 0 基础版独赢: 1
base 通过: 6 / 8 95%精确区间: (0.349, 0.968)
condition_check 通过: 5 / 8 95%精确区间: (0.245, 0.915)
不一致对的双侧符号检验 p: 1.0


**结果解读**

逐题配对保留了同一任务上的得失。精确区间与 p 值描述这组小样本，不消除评分器盲区、任务集偏差或模型非确定性。

**小练习**

两个版本总通过数相同，配对表仍可能揭示什么？

<details><summary>完成后展开参考分析</summary>

可能在不同题目上各有赢输，均值会抹掉回归。应按任务和失败层查看不一致对。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

In [5]:
def diagnose_format(task,trial):
    """只作为复盘：从唯一 JSON 围栏提取内容，不改变原始格式失败判定。"""
    import copy,re
    blocks=re.findall(r'```json\s*\n(.*?)\n```',trial['state']['answer'],flags=re.S)
    if len(blocks)!=1:return {'available':False,'reason':'没有唯一完整 JSON 围栏'}
    extracted=copy.deepcopy(trial);extracted['state']['answer']=blocks[0]
    grade=grade_trial(task,extracted)
    return {'available':True,'extracted_grade':grade,
            'note':'诊断提取后事实与引用检查；不豁免原任务的输出格式要求，仍需核验围栏之外的文字。'}


In [6]:
diagnostics = []
for trial in record["trials"]:
    diagnostic = diagnose_format(TASKS[trial["task_id"]],trial)
    diagnostics.append(diagnostic)
    if diagnostic["available"]:
        print(trial["task_id"],trial["variant"],diagnostic["extracted_grade"]["checks"])
print("这是事后诊断。严格任务分数没有因提取而被改写。")


missing-file base {'completed': True, 'schema': False, 'facts': False, 'citations': False, 'source_unchanged': True}
json-contract condition_check {'completed': True, 'schema': True, 'facts': False, 'citations': True, 'source_unchanged': True}
absolute-env base {'completed': True, 'schema': True, 'facts': True, 'citations': True, 'source_unchanged': True}
absolute-env base {'completed': True, 'schema': True, 'facts': True, 'citations': True, 'source_unchanged': True}
这是事后诊断。严格任务分数没有因提取而被改写。


### 逐题配对看改动，保留失败在分母里

配套课件：L07.06-S01、L07.06-S02、L07.07-S02。

总分可能掩盖某道题变好、另一道变差。按 task_id 和 repeat 把两个版本的原始试验放在同一行，同时显示停止原因、严格评分和诊断是否可用。诊断只帮助归因，不覆盖原始分数。

In [7]:
case_pairs = defaultdict(dict)
case_grades_before = [t["grade"]["passed"] for t in record["trials"]]
for case_trial in record["trials"]:
    case_key = (case_trial["task_id"], case_trial["repeat"])
    case_pairs[case_key][case_trial["variant"]] = {
        "status": case_trial["state"]["status"], "strict_pass": case_trial["grade"]["passed"],
        "diagnostic_available": diagnose_format(TASKS[case_trial["task_id"]], case_trial)["available"]}
for case_key, case_pair in sorted(case_pairs.items()):
    print("任务 / 次数:", case_key)
    for case_variant, case_row in sorted(case_pair.items()):
        print("  ", case_variant, "停止:", case_row["status"], "严格通过:", case_row["strict_pass"], "诊断可用:", case_row["diagnostic_available"])
assert sum(len(pair) for pair in case_pairs.values()) == len(record["trials"])
assert case_grades_before == [t["grade"]["passed"] for t in record["trials"]]


任务 / 次数: ('absolute-env', 1)
   base 停止: final_answer 严格通过: True 诊断可用: True
   condition_check 停止: final_answer 严格通过: True 诊断可用: False
任务 / 次数: ('absolute-env', 2)
   base 停止: final_answer 严格通过: True 诊断可用: True
   condition_check 停止: final_answer 严格通过: True 诊断可用: False
任务 / 次数: ('json-contract', 1)
   base 停止: final_answer 严格通过: True 诊断可用: False
   condition_check 停止: final_answer 严格通过: False 诊断可用: True
任务 / 次数: ('json-contract', 2)
   base 停止: final_answer 严格通过: True 诊断可用: False
   condition_check 停止: final_answer 严格通过: True 诊断可用: False
任务 / 次数: ('missing-file', 1)
   base 停止: final_answer 严格通过: False 诊断可用: True
   condition_check 停止: final_answer 严格通过: False 诊断可用: False
任务 / 次数: ('missing-file', 2)
   base 停止: final_answer 严格通过: False 诊断可用: False
   condition_check 停止: tool_budget 严格通过: False 诊断可用: False
任务 / 次数: ('relative-base', 1)
   base 停止: final_answer 严格通过: True 诊断可用: False
   condition_check 停止: final_answer 严格通过: True 诊断可用: False
任务 / 次数: ('relative-base', 2)
   base 停止: fin

**结果解读**

表中的格式失败仍是严格任务失败；即使提取围栏后内容可评分，也不能事后当成通过。如果某一侧缺失，这一对就不完整，要报告缺失而不是删除失败样本。

**小练习**

挑一行，给下一轮实验提出一个改动，并列出至少两项要固定的条件。

<details><summary>完成后展开参考分析</summary>

例如只修改输出格式提示，固定任务版本、模型设置、预算和评分器；重新运行后的记录另存，不覆盖本轮原始回答。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

## 3. 聚合不能抹掉失败原因

同一版本先按任务分组，再汇总。下面的 Token 字段采用返回过的 usage，是已知用量；异常请求可能存在本地未记录用量。耗时包含本地 MCP 进程和网络，不是纯模型推理耗时。

如果两个版本在严格任务要求下都很低，先检查具体失败项。不要为了证明提示有效而改用只对其中一个版本有利的统计口径。


In [8]:
def summarize(trials,k=1):
    """按 variant 和 task 聚合；失败试验仍留在分母与资源统计中。"""
    variants=defaultdict(lambda:defaultdict(list))
    for trial in trials: variants[trial['variant']][trial['task_id']].append(trial)
    rows=[]
    for variant,tasks in sorted(variants.items()):
        per_task={name:pass_metrics([t['grade']['passed'] for t in runs],k) for name,runs in tasks.items()}
        flat=[t for runs in tasks.values() for t in runs]
        rows.append({'variant':variant,'tasks':len(tasks),'trials':len(flat),
            'passed_trials':sum(t['grade']['passed'] for t in flat),
            'macro_pass_at_k':statistics.mean(v['pass_at_k'] for v in per_task.values()),
            'macro_pass_power_k':statistics.mean(v['pass_power_k'] for v in per_task.values()),
            'known_total_tokens':sum(sum(u.get('total_tokens',0) for u in t.get('observed_request_usage',t['state']['usage'])) for t in flat),
            'usage_complete':all(len(t.get('observed_request_usage',t['state']['usage']))==t['state']['turns'] for t in flat),
            'model_requests':sum(t['state']['turns'] for t in flat),
            'tool_calls':sum(t['state']['tool_calls'] for t in flat),
            'mean_elapsed_seconds':statistics.mean(t['elapsed_seconds'] for t in flat),'per_task':per_task})
    return rows


In [9]:
for row in summarize(record["trials"],1):
    print({key:value for key,value in row.items() if key!="per_task"})
if record["complete"]:
    for row in summarize(record["trials"],2):
        print(row["variant"], "k=2:", row["macro_pass_at_k"],row["macro_pass_power_k"])
else:
    print("计划未全部执行，不计算完整配对的 k=2 对照。")


{'variant': 'base', 'tasks': 4, 'trials': 8, 'passed_trials': 6, 'macro_pass_at_k': 0.75, 'macro_pass_power_k': 0.75, 'known_total_tokens': 45077, 'usage_complete': True, 'model_requests': 24, 'tool_calls': 48, 'mean_elapsed_seconds': 16.806125}
{'variant': 'condition_check', 'tasks': 4, 'trials': 8, 'passed_trials': 5, 'macro_pass_at_k': 0.625, 'macro_pass_power_k': 0.625, 'known_total_tokens': 45500, 'usage_complete': True, 'model_requests': 24, 'tool_calls': 50, 'mean_elapsed_seconds': 17.196575}
base k=2: 0.75 0.75
condition_check k=2: 0.75 0.5


## 练习：给失败做可核查归因

选择一条真实试验，列出运行状态、输出格式、结构事实、引用、自然语言五个维度。指出哪些已经查证，哪些仍需要审阅。

然后提出下一次只改一项的实验，例如为结构输出增加专门的格式约束。此时应保留旧实验，再建立新的实验版本与任务划分，不覆盖已有记录。


In [10]:
selected = record["trials"][0]
worksheet = {"task":selected["task_id"],"run_id":selected["state"]["run_id"],
    "status":selected["state"]["status"],"strict_checks":selected["grade"]["checks"],
    "content_diagnostic":diagnose_format(TASKS[selected["task_id"]],selected),
    "human_review":"待独立核对整段解释，包括 JSON 围栏外的文字"}
print(json.dumps(worksheet,ensure_ascii=False,indent=2))


{
  "task": "relative-base",
  "run_id": "02e9f9ae29b64e7080d88cc683264264",
  "status": "final_answer",
  "strict_checks": {
    "completed": true,
    "schema": true,
    "facts": true,
    "citations": true,
    "source_unchanged": true
  },
  "content_diagnostic": {
    "available": false,
    "reason": "没有唯一完整 JSON 围栏"
  },
  "human_review": "待独立核对整段解释，包括 JSON 围栏外的文字"
}


In [11]:
assert worksheet["run_id"] == selected["state"]["run_id"]
print("先定位可观察失败，再提出一次改动；不要仅用一个总分代替失败分析。")


先定位可观察失败，再提出一次改动；不要仅用一个总分代替失败分析。


## 可带走的结果：受控对照与失败诊断报告

最后把唯一改动、固定条件、每个版本的任务级指标、试验状态和一条失败诊断写成 JSON 与 Markdown。原始严格分数不因事后提取而改写，缺失请求也不会从分母中消失。


In [12]:
report_dir = ROOT / ".local/student-runs/l07"
report_dir.mkdir(parents=True, exist_ok=True)
summary_k1 = summarize(record["trials"], 1)
summary_k2 = summarize(record["trials"], 2) if record["complete"] else []
status_by_variant = defaultdict(lambda: defaultdict(int))
for trial in record["trials"]:
    status_by_variant[trial["variant"]][trial["state"]["status"]] += 1
report = {
    "report_type": "controlled-comparison-and-failure-diagnosis",
    "recorded_at": record["recorded_at"],
    "run_model": record["model"],
    "single_change": record["single_change"],
    "fixed_conditions": {
        "dataset_sha256": record["dataset_sha256"],
        "source_snapshot": record["source_snapshot"],
        "limits": record["limits"],
        "max_tokens_per_call": record["max_tokens_per_call"],
        "automatic_retries": record["automatic_retries"],
    },
    "trial_count": len(record["trials"]),
    "complete": record["complete"],
    "status_by_variant": {variant: dict(counts) for variant, counts in status_by_variant.items()},
    "summary_k1": summary_k1,
    "summary_k2": summary_k2,
    "failure_worksheet": worksheet,
    "diagnostic_rule": "Post-hoc extraction may explain a format failure, but never rewrites the original strict score.",
    "next_experiment": {
        "single_change": "add one dedicated structured-output constraint",
        "hold_fixed": ["task set", "source snapshot", "model", "tools", "limits", "graders"],
    },
}
json_path = report_dir / "controlled-comparison-report.json"
md_path = report_dir / "controlled-comparison-report.md"
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
md_path.write_text(
    "# 受控对照与失败诊断报告\n\n"
    f"- 本次模型：`{report['run_model']}`\n"
    f"- 唯一改动：{report['single_change']}\n"
    f"- 试验数量：{report['trial_count']}\n"
    f"- 计划完整：{report['complete']}\n"
    f"- 失败样例：`{worksheet['task']}` / `{worksheet['run_id']}`\n\n"
    "> 事后诊断用于解释失败，不覆盖原始严格评分；未来重跑前先查询当前账号可用模型。\n",
    encoding="utf-8",
)
print("已保存对照报告:", json_path.relative_to(ROOT), "和", md_path.relative_to(ROOT))
print(json.dumps(report, ensure_ascii=False, indent=2))


已保存对照报告: .local\student-runs\l07\controlled-comparison-report.json 和 .local\student-runs\l07\controlled-comparison-report.md
{
  "report_type": "controlled-comparison-and-failure-diagnosis",
  "recorded_at": "2026-09-24T05:47:54.029380+00:00",
  "run_model": "deepseek-flash",
  "single_change": "回答前逐项核对运行条件：区分源码保证与样例文件；区分相对路径与绝对路径；没有检查的类型和字段不要声称有保证。",
  "fixed_conditions": {
    "dataset_sha256": "e6ace50ed89586612df164f72ce57804ea63731d639ae64a8d95475f3ecc1c8f",
    "source_snapshot": {
      "02 从大模型到 Agent 组成与最小循环/数据/demo-repo/config.py": "98dfc722dd0cdda2276276af59ea3c239bf4e218a5721712e2057817bd702979",
      "02 从大模型到 Agent 组成与最小循环/数据/demo-repo/main.py": "1482d39ac1a0ad5724f5a0700c60c82d6cd48db71d8960edc849f0c5ccc03b3f",
      "02 从大模型到 Agent 组成与最小循环/数据/demo-repo/README.md": "b45296e59f38a0d03904c56030dd147b5f5023f31a77a3aa39a5a8379f76102e",
      "02 从大模型到 Agent 组成与最小循环/数据/demo-repo/settings.json": "32a0d938a933c4fa3bb1467fe87a779944fcee4f4b74d314ce898c01abaff879",
      "05 MCP

## 本节学会了什么

定义任务与成功标准；独立执行并保留轨迹；组合规则、开放语义核验与独立人工复核；区分 pass@k 和 pass^k；用控制变量和失败分析推动下一次改进。

下一节把评估用于资料与仓库学习助手。来源：[模型评分偏差研究](https://arxiv.org/abs/2306.05685)、[Agent 评估实践](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)。
